# Setup
## Imports

# Pipeline Object

In [ ]:
# Imports
import json
import os
import re
import time
from abc import ABC, abstractmethod
from collections import Counter
from itertools import chain

import numpy as np
import pandas as pd
import psutil
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import (
    col,
    collect_list,
    expr,
    lag,
    pandas_udf,
    regexp_extract,
    row_number,
    to_timestamp,
    udf,
    unix_timestamp,
    when,
)
from pyspark.sql.types import ArrayType, StringType
from utils import check_parseDF, print_section

# Create Spark Session with configurations for M1 Max
spark = (
    SparkSession.builder.appName("HDFS Log Processing")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.executor.memory", "4g")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)


# Column patterns for regex extraction
COL_PATTERNS = {
    "date_time": r"^(\d{6} \d{6})",
    "pid": r"^(?:\d{6} \d{6}) (\d+)",
    "level": r"^(?:\d{6} \d{6}) (?:\d+) (\w+)",
    "component": r"^(?:\d{6} \d{6}) (?:\d+) (?:\w+) ([^:]+)",
    "BlockId": r"(blk_-?\d+)",
}


# Base abstract class
class Pipe(ABC):
    def __init__(self, data_dir="data/"):
        self.data_dir = data_dir
        self.metrics = {}
        self.logs = None
        self.anomalies_df = None
        self.actual_events = None
        self.event_markers = None

    def display_metrics(self):
        print(
            json.dumps(
                {
                    k: round(v, 4) if isinstance(v, (int, float)) else v
                    for k, v in self.metrics.items()
                },
                indent=4,
            )
        )

    @staticmethod
    def mem_metrics(metrics_dict, stage_name):
        mem = psutil.virtual_memory()
        metrics_dict[f"{stage_name}_gbtot"] = mem.total / (1024**3)
        metrics_dict[f"{stage_name}_gbused"] = mem.used / (1024**3)
        metrics_dict[f"{stage_name}_mempct"] = mem.percent

    def get_event_markers(self, spark):
        """Get event markers dictionary from templates file."""
        templates_df = (
            spark.read.format("csv")
            .option("inferSchema", "true")
            .option("header", "true")
            .load(os.path.join(self.data_dir, "HDFS.log_templates.csv"))
        )

        # Extract words from templates
        def extract_words(template):
            words = [
                w
                for part in re.split(r"\[\*\]", template)
                for w in re.findall(r"[\w\.:]+", part)
                if w
            ]
            return ["BLOCK*" if w == "BLOCK" else w for w in words]

        # Extract markers from each template
        template_rows = templates_df.select("EventId", "EventTemplate").collect()
        self.event_markers = {
            row["EventId"]: extract_words(row["EventTemplate"]) for row in template_rows
        }

        return self.event_markers

    @abstractmethod
    def load(self):
        """Load data from source."""
        pass

    @abstractmethod
    def parse(self):
        """Parse loaded data."""
        pass

    @abstractmethod
    def group(self, mode=True, partition_strategy=None):
        """Group data into meaningful chunks."""
        pass

    @abstractmethod
    def pipeline(self):
        """Execute full data processing pipeline."""
        pass


# Concrete implementation
class PipeDF(Pipe):
    def __init__(self, spark, data_dir="data/"):
        super().__init__(data_dir)
        self.spark = spark
        self.parsed_df = None
        self.blocks_df = None

    def load(self):
        start = time.time()
        print_section("STEP 1: Loading Data")

        # Get event markers
        self.get_event_markers(self.spark)

        # Load logs
        self.logs = self.spark.read.text(os.path.join(self.data_dir, "HDFS.log"))

        # Load events from NPZ file
        self.actual_events = dict(
            Counter(
                chain.from_iterable(
                    np.load(os.path.join(self.data_dir, "HDFS.npz"), allow_pickle=True)[
                        "x_data"
                    ]
                )
            )
        )

        # Load anomalies
        self.anomalies_df = (
            self.spark.read.format("csv")
            .option("inferSchema", "true")
            .option("header", "true")
            .load(os.path.join(self.data_dir, "anomaly_label.csv"))
            .withColumn(
                "Label",
                when(col("Label") == "Normal", 0)
                .when(col("Label") == "Anomaly", 1)
                .otherwise(col("Label")),
            )
        )

        self.logs.show(3, truncate=False)
        self.metrics["df_load_time"] = time.time() - start
        self.mem_metrics(self.metrics, "df_load")

        return self.logs, self.anomalies_df, self.actual_events

    def parse(self):
        start = time.time()
        print_section("STEP 2: Parse Logs to Extract and Encode Fields")

        # Convert event templates to dictionary for easy lookup in UDF
        bc_event_markers = self.spark.sparkContext.broadcast(self.event_markers)

        @pandas_udf(StringType())
        def assign_event(series: pd.Series) -> pd.Series:
            """Match log messages to event markers from templates."""
            results = []

            for log_message in series:
                # Default to unknown
                event_id = "Unknown"

                # Try to match each event's marker words
                for event, markers in bc_event_markers.value.items():
                    if all(log_message.find(marker) != -1 for marker in markers):
                        event_id = event
                        break

                results.append(event_id)

            return pd.Series(results)

        # Parse log data and set directly to class attribute
        self.parsed_df = (
            self.logs.withColumn("EventId", assign_event(col("value")))
            # Add all regex extracted columns
            .select(
                *[
                    regexp_extract(col("value"), pattern, 1).alias(field)
                    for field, pattern in COL_PATTERNS.items()
                ],
                "value",
                "EventId",
            )
            # Add timestamp from extracted date
            .withColumn("timestamp", to_timestamp(col("date_time"), "yyMMdd HHmmss"))
        )

        # Run validation checks
        check_parseDF(self.parsed_df, self.actual_events)

        # Show sample data
        self.parsed_df.show(5, truncate=False)

        # Record metrics
        self.metrics["df_parse_time"] = time.time() - start
        self.mem_metrics(self.metrics, "df_parse")

        return self.parsed_df

    def group(self, mode=True, partition_strategy=None):
        start = time.time()

        # Add sequence numbers and calculate intervals in chained operations
        processed_df = (
            self.parsed_df.withColumn(
                "seq_num",
                row_number().over(Window.partitionBy("BlockId").orderBy("timestamp")),
            )
            .withColumn(
                "prev_timestamp",
                lag("timestamp", 1).over(
                    Window.partitionBy("BlockId").orderBy("seq_num")
                ),
            )
            .withColumn(
                "interval_seconds",
                when(col("prev_timestamp").isNull(), 0).otherwise(
                    unix_timestamp("timestamp") - unix_timestamp("prev_timestamp")
                ),
            )
        )

        # Group by block ID and collect data
        block_events = (
            processed_df.groupBy("BlockId")
            .agg(
                collect_list(expr("struct(seq_num, EventId, interval_seconds)")).alias(
                    "event_data"
                ),
                collect_list("level").alias("levels"),
                collect_list("component").alias("components"),
                collect_list("pid").alias("pids"),
            )
            # Extract events and intervals from structured data
            .withColumn(
                "events",
                expr("TRANSFORM(SORT_ARRAY(event_data, true), x -> x.EventId)"),
            )
            .withColumn(
                "intervals",
                expr(
                    "TRANSFORM(SORT_ARRAY(event_data, true), x -> x.interval_seconds)"
                ),
            )
            # Calculate total latency (sum of all intervals)
            .withColumn(
                "latency",
                expr("aggregate(intervals, CAST(0 as BIGINT), (acc, x) -> acc + x)"),
            )
            # Join with anomaly labels - using simple equijoin syntax
            .join(self.anomalies_df, "BlockId", "left")
        )

        # Store final result with selected columns
        self.blocks_df = block_events.select(
            "BlockId",
            "events",
            "intervals",
            "latency",
            "levels",
            "components",
            "pids",
            "Label",
        )

        # Display and record metrics
        self.blocks_df.show(5, truncate=True)
        self.metrics["df_group_time"] = time.time() - start
        self.mem_metrics(self.metrics, "df_group")

        return self.blocks_df

    def pipeline(self):
        """Run the full data processing pipeline."""
        self.load()
        self.parse()
        self.group()
        return self


class PipeRDD(Pipe):
    def __init__(self, spark, data_dir="data/"):
        super().__init__(data_dir)
        self.spark = spark

        # Data containers
        self.logs_rdd = None
        self.anomalies_rdd = None
        self.actual_events = None
        self.parsed_rdd = None
        self.blocks_rdd = None
        self.event_markers = None

    def load(self):
        start_time = time.time()
        print_section("STEP 1: Loading Data (RDD Implementation)")

        # Get event markers
        self.get_event_markers(self.spark)

        # Load log file as RDD
        self.logs_rdd = self.spark.sparkContext.textFile(
            os.path.join(self.data_dir, "HDFS.log")
        )

        # Load actual events from NPZ file
        self.actual_events = dict(
            Counter(
                chain.from_iterable(
                    np.load(os.path.join(self.data_dir, "HDFS.npz"), allow_pickle=True)[
                        "x_data"
                    ]
                )
            )
        )

        # Load anomalies CSV and convert to RDD
        anomalies_df = (
            self.spark.read.format("csv")
            .option("inferSchema", "true")
            .option("header", "true")
            .load(os.path.join(self.data_dir, "anomaly_label.csv"))
        )

        # Convert DataFrame to RDD with (BlockId, Label) pairs
        self.anomalies_rdd = anomalies_df.rdd.map(
            lambda row: (row["BlockId"], 1 if row["Label"] == "Anomaly" else 0)
        )

        # Show first few log entries
        print("First 3 Log Entries:")
        for line in self.logs_rdd.take(3):
            print(line)

        # Record loading metrics
        self.metrics["rdd_load_time"] = time.time() - start_time
        self.mem_metrics(self.metrics, "rdd_load")

        return self.logs_rdd, self.anomalies_rdd, self.actual_events

    def parse(self):
        start = time.time()
        print_section(
            "STEP 2: Parse Logs to Extract and Encode Fields (RDD Implementation)"
        )

        # Broadcast patterns for efficient access across nodes
        bc_event_markers = self.spark.sparkContext.broadcast(self.event_markers)
        bc_col_patterns = self.spark.sparkContext.broadcast(COL_PATTERNS)

        def parse_log_line(line):
            """Parse a single log line into components"""
            # Extract fields using regex patterns
            fields = {}
            for field, pattern in bc_col_patterns.value.items():
                match = re.search(pattern, line)
                fields[field] = match.group(1) if match else ""

            # Determine event type by matching markers
            event_id = "Unknown"
            for e_id, markers in bc_event_markers.value.items():
                if all(line.find(marker) != -1 for marker in markers):
                    event_id = e_id
                    break

            # Return the parsed log entry
            return (
                fields["BlockId"],
                fields["date_time"],
                fields["pid"],
                fields["level"],
                fields["component"],
                event_id,
                line,  # Keep original line for reference
            )

        # Apply parsing to logs and filter out entries with missing blockIDs
        parsed_rdd = self.logs_rdd.map(parse_log_line).filter(lambda x: x[0])

        # RDD Structure: (BlockId, date_time, pid, level, component, event_id, raw_line)
        self.parsed_rdd = parsed_rdd

        # Display sample of parsed data
        # print("\nSample of parsed RDD data:")
        # print(
        #     "+------------------------+-------------+-----+-------+----------------------------+-------+"
        # )
        # print(
        #     "|BlockId                 |date_time    |pid  |level  |component                   |EventId|"
        # )
        # print(
        #     "+------------------------+-------------+-----+-------+----------------------------+-------+"
        # )
        # for row in self.parsed_rdd.take(5):
        #     blk_id, date_time, pid, level, component, event_id, _ = row
        #     print(
        #         f"|{blk_id:<24}|{date_time:<13}|{pid:<5}|{level:<7}|{component:<28}|{event_id:<7}|"
        #     )
        # print(
        #     "+------------------------+-------------+-----+-------+----------------------------+-------+"
        # )

        # Record metrics
        self.metrics["rdd_parse_time"] = time.time() - start
        self.mem_metrics(self.metrics, "rdd_parse")

        return self.parsed_rdd

    def group(self, mode=True, partition_strategy=None):
        start = time.time()
        # print_section("STEP 3: GroupBy BlockID (RDD Implementation)")

        # Apply partitioning strategy if provided
        working_rdd = self.parsed_rdd

        # Convert date_time to timestamp for sorting
        def to_timestamp(date_time_str):
            if not date_time_str:
                return None
            try:
                # Format: "yyMMdd HHmmss"
                from datetime import datetime

                return datetime.strptime(date_time_str, "%y%m%d %H%M%S")
            except:
                return None

        # Add timestamp to each record
        timestamped_rdd = working_rdd.map(
            lambda x: (
                x[0],  # BlockId
                x[5],  # EventId
                to_timestamp(x[1]),  # timestamp
                x[2],  # pid
                x[3],  # level
                x[4],  # component
                x[6],  # raw_line
            )
        )

        # Group by BlockId
        grouped_rdd = timestamped_rdd.groupBy(lambda x: x[0])

        # Process each block group to calculate intervals, etc.
        def process_block_group(block_group):
            block_id, events_iter = block_group

            # Convert iterator to list for multiple passes
            events_list = list(events_iter)

            # Sort events by timestamp
            sorted_events = sorted(
                events_list, key=lambda x: x[2] if x[2] else datetime.min
            )

            # Extract data
            events = [e[1] for e in sorted_events]  # EventIds
            pids = [e[3] for e in sorted_events]  # PIDs
            levels = [e[4] for e in sorted_events]  # Levels
            components = [e[5] for e in sorted_events]  # Components

            # Calculate intervals
            intervals = [0]  # First event has no interval
            latency = 0
            for i in range(1, len(sorted_events)):
                current_time = sorted_events[i][2]
                prev_time = sorted_events[i - 1][2]

                if current_time and prev_time:
                    interval = int((current_time - prev_time).total_seconds())
                else:
                    interval = 0

                intervals.append(interval)
                latency += interval

            return (block_id, events, intervals, latency, levels, components, pids)

        processed_blocks = grouped_rdd.map(process_block_group)

        # Join with anomaly labels
        def join_with_labels(block_data, anomalies_list):
            block_id, events, intervals, latency, levels, components, pids = block_data

            # Find matching anomaly label or default to 0 (normal)
            label = 0
            for a_id, a_label in anomalies_list:
                if a_id == block_id:
                    label = a_label
                    break

            return (
                block_id,
                events,
                intervals,
                latency,
                levels,
                components,
                pids,
                label,
            )

        # Collect anomalies for local joining (more efficient than distributed join for this case)
        anomalies_list = self.anomalies_rdd.collect()
        bc_anomalies = self.spark.sparkContext.broadcast(anomalies_list)

        # Join blocks with anomaly labels
        self.blocks_rdd = processed_blocks.map(
            lambda block: join_with_labels(block, bc_anomalies.value)
        )

        # Show sample results
        # print("\nSample of grouped block data:")
        # print(
        #     "+------------------------+--------------------+--------------------+-------+--------------------+-----+"
        # )
        # print(
        #     "|BlockId                 |events              |intervals           |latency|components          |Label|"
        # )
        # print(
        #     "+------------------------+--------------------+--------------------+-------+--------------------+-----+"
        # )
        # for block in self.blocks_rdd.take(5):
        #     block_id, events, intervals, latency, levels, components, pids, label = (
        #         block
        #     )
        #     print(
        #         f"|{block_id:<24}|{str(events[:3]):<20}|{str(intervals[:3]):<20}|{latency:<7}|{str(components[:3]):<20}|{label:<5}|"
        #     )
        # print(
        #     "+------------------------+--------------------+--------------------+-------+--------------------+-----+"
        # )

        # Record metrics
        self.metrics["rdd_group_time"] = time.time() - start
        self.mem_metrics(self.metrics, "rdd_group")

        return self.blocks_rdd

    def pipeline(self):
        """Run the full RDD processing pipeline."""
        self.load()
        self.parse()
        self.group()
        return self

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/12 12:37:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 60720)
Traceback (most recent call last):
  File "/Users/farzanmirza/miniconda3/envs/pyspark_env/lib/python3.9/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/Users/farzanmirza/miniconda3/envs/pyspark_env/lib/python3.9/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/Users/farzanmirza/miniconda3/envs/pyspark_env/lib/python3.9/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/Users/farzanmirza/miniconda3/envs/pyspark_env/lib/python3.9/socketserver.py", line 747, in __init__
    self.handle()
  File "/Users/farzanmirza/miniconda3/envs/pyspark_env/lib/python3.9/site-packages/pyspark/accumulators.py", line 281, in handle
    poll(accum_updates)
  File "/Users/farzanmirza/miniconda3/envs/pyspark_env/l

In [ ]:
# Create PipeDF instance
df_pipe = PipeDF(spark=spark)
df_pipe.pipeline()
df_pipe.display_metrics()
# Create PipeRDD instance
rdd_pipe = PipeRDD(spark=spark)
rdd_pipe.pipeline()
rdd_pipe.display_metrics()


STEP 1: Loading Data
+----------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                     |
+----------------------------------------------------------------------------------------------------------------------------------------------------------+
|081109 203518 143 INFO dfs.DataNode$DataXceiver: Receiving block blk_-1608999687919862906 src: /10.250.19.102:54106 dest: /10.250.19.102:50010            |
|081109 203518 35 INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: /mnt/hadoop/mapred/system/job_200811092030_0001/job.jar. blk_-1608999687919862906|
|081109 203519 143 INFO dfs.DataNode$DataXceiver: Receiving block blk_-1608999687919862906 src: /10.250.10.6:40524 dest: /10.250.10.6:50010                |
+-----------------------------------

+-------------+---+-----+----------------------------+------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------+-------+-------------------+
|date_time    |pid|level|component                   |BlockId                 |value                                                                                                                                                     |EventId|timestamp          |
+-------------+---+-----+----------------------------+------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------+-------+-------------------+
|081109 203518|143|INFO |dfs.DataNode$DataXceiver    |blk_-1608999687919862906|081109 203518 143 INFO dfs.DataNode$DataXceiver: Receiving block blk_-1608999687919862906 src: /10.250.19.102:54106 dest: /10.250.19

OpenJDK 64-Bit Server VM warning: CodeCache is full. Compiler has been disabled.
OpenJDK 64-Bit Server VM warning: Try increasing the code cache size using -XX:ReservedCodeCacheSize=


CodeCache: size=131072Kb used=30982Kb max_used=31002Kb free=100089Kb
 bounds [0x000000010798c000, 0x000000010980c000, 0x000000010f98c000]
 total_blobs=11767 nmethods=10798 adapters=880
 compilation: disabled (not enough contiguous free space left)


+--------------------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+-----+
|             BlockId|              events|           intervals|latency|              levels|          components|                pids|Label|
+--------------------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+-----+
|blk_-100009528570...|[E22, E5, E5, E5,...|[0, 0, 0, 0, 34, ...|  36773|[INFO, INFO, INFO...|[dfs.FSNamesystem...|[34, 9453, 9461, ...|    0|
|blk_-100021874837...|[E5, E5, E5, E22,...|[0, 0, 0, 0, 26, ...|  23425|[INFO, INFO, INFO...|[dfs.DataNode$Dat...|[12321, 12447, 12...|    0|
|blk_-100072357794...|   [E5, E22, E5, E7]|        [0, 0, 2, 2]|      4|[INFO, INFO, INFO...|[dfs.DataNode$Dat...|[24114, 31, 24263...|    1|
|blk_-100077559819...|[E22, E5, E5, E5,...|[0, 0, 0, 0, 43, ...|  31266|[INFO, INFO, INFO...|[dfs.FSNamesystem...|[28, 6021, 6051, ...|    0|
|blk_-

|blk_-9073992586687739851|['E5', 'E22', 'E5'] |[0, 0, 1]           |50448  |['dfs.DataNode$DataXceiver', 'dfs.FSNamesystem', 'dfs.DataNode$DataXceiver']|0    |
|blk_-3409923645141256069|['E5', 'E22', 'E5'] |[0, 0, 1]           |50443  |['dfs.DataNode$DataXceiver', 'dfs.FSNamesystem', 'dfs.DataNode$DataXceiver']|0    |
|blk_-2398209593415798905|['E5', 'E5', 'E22'] |[0, 0, 0]           |50446  |['dfs.DataNode$DataXceiver', 'dfs.DataNode$DataXceiver', 'dfs.FSNamesystem']|0    |
|blk_4521164666406024155 |['E5', 'E5', 'E22'] |[0, 0, 0]           |50464  |['dfs.DataNode$DataXceiver', 'dfs.DataNode$DataXceiver', 'dfs.FSNamesystem']|0    |
|blk_-8515113585876695879|['E5', 'E5', 'E22'] |[0, 0, 0]           |50333  |['dfs.DataNode$DataXceiver', 'dfs.DataNode$DataXceiver', 'dfs.FSNamesystem']|0    |
+------------------------+--------------------+--------------------+-------+--------------------+-----+
{
    "rdd_load_time": 2.0528,
    "rdd_load_gbtot": 32.0,
    "rdd_load_gbused": 14.3474,
    "

In [ ]:
# Add this to your notebook
import time

import psutil
from pyspark.sql import SparkSession


class PerformanceTester:

    def __init__(self, pipe_instance, spark):

        self.pipe = pipe_instance
        self.spark = spark
        self.test_metrics = {}
        self.pipe_type = "df" if isinstance(pipe_instance, PipeDF) else "rdd"

    def mem_metrics(self, stage_name):
        """Record memory metrics for a specific test stage."""
        mem = psutil.virtual_memory()
        self.test_metrics[f"{stage_name}_gbtot"] = mem.total / (1024**3)
        self.test_metrics[f"{stage_name}_gbused"] = mem.used / (1024**3)
        self.test_metrics[f"{stage_name}_mempct"] = mem.percent

    def display_metrics(self):
        """Display test metrics in a formatted way."""
        import json

        print(
            json.dumps(
                {
                    k: round(v, 4) if isinstance(v, (int, float)) else v
                    for k, v in self.test_metrics.items()
                },
                indent=4,
            )
        )

    def test_partition_count(self):
        """Test 1: Performance with different partition counts."""
        print_section(f"TEST 1: Partition Count Testing ({self.pipe_type.upper()})")
        t1_start_time = time.time()

        # Loop over partition counts
        for num_partitions in [1, 2, 4, 8, 16, 32, 64]:
            self.spark.conf.set("spark.sql.shuffle.partitions", num_partitions)
            start_time_part = time.time()

            if self.pipe_type == "df":
                # For DataFrame implementation - suppress output with mode=False
                result = self.pipe.group(mode=False).count()
            else:
                # For RDD implementation
                result = self.pipe.group(mode=False)
                # Trigger evaluation for timing
                if hasattr(result, "count"):
                    result.count()

            # Store metrics
            self.test_metrics[f"t1_p{num_partitions}"] = time.time() - start_time_part
            print(
                f"  Partitions: {num_partitions}, Time: {self.test_metrics[f't1_p{num_partitions}']:.4f} seconds"
            )

        # Reset to default partition count
        self.spark.conf.set("spark.sql.shuffle.partitions", "8")

        # Record total time for this test
        self.test_metrics["t1_tot"] = time.time() - t1_start_time
        self.mem_metrics("t1")

        print(f"Total test time: {self.test_metrics['t1_tot']:.4f} seconds\n")
        return self.test_metrics

    def test_repartitioning(self):
        """Test 2: Performance impact of repartitioning at different pipeline stages."""
        print_section(f"TEST 2: Repartitioning Stages ({self.pipe_type.upper()})")
        t2_start_time = time.time()

        # Use a consistent partition count for fair comparison
        partition_count = 16

        # Test 1: No repartitioning (baseline)
        start_time = time.time()
        if self.pipe_type == "df":
            self.pipe.group(mode=False).count()
        else:
            result = self.pipe.group(mode=False)
            if hasattr(result, "count"):
                result.count()
        self.test_metrics["t2_base"] = time.time() - start_time
        print(
            f"  Baseline (no repartitioning): {self.test_metrics['t2_base']:.4f} seconds"
        )

        # Test 2: Repartition after parsing
        start_time = time.time()
        if self.pipe_type == "df":
            # Save original for later
            orig_df = self.pipe.parsed_df
            self.pipe.parsed_df = orig_df.repartition(partition_count)
            # Run the group operation
            self.pipe.group(mode=False).count()
            # Restore original
            self.pipe.parsed_df = orig_df
        else:
            # For RDD implementation
            if hasattr(self.pipe, "parsed_rdd"):
                orig_rdd = self.pipe.parsed_rdd
                self.pipe.parsed_rdd = orig_rdd.repartition(partition_count)
                result = self.pipe.group(mode=False)
                if hasattr(result, "count"):
                    result.count()
                self.pipe.parsed_rdd = orig_rdd

        self.test_metrics["t2_repart"] = time.time() - start_time
        print(
            f"  Repartition after parsing: {self.test_metrics['t2_repart']:.4f} seconds"
        )

        # Record total test time
        self.test_metrics["t2_tot"] = time.time() - t2_start_time
        self.mem_metrics("t2")

        print(f"Total test time: {self.test_metrics['t2_tot']:.4f} seconds\n")
        return self.test_metrics

    def test_coalesce_vs_repartition(self):
        """Test 3: Performance comparison between coalesce and repartition operations."""
        print_section(f"TEST 3: Coalesce vs Repartition ({self.pipe_type.upper()})")
        t3_start_time = time.time()

        # Configuration parameters
        high_part, low_part = 32, 8

        # Test configs
        test_configs = [
            {
                "name": "base",
                "desc": "No partitioning (baseline)",
                "transform": lambda df: df,
            },
            {
                "name": "repup",
                "desc": "Repartition to increase partitions",
                "transform": lambda df: df.repartition(high_part),
            },
            {
                "name": "repdown",
                "desc": "Repartition to decrease partitions",
                "transform": lambda df: df.repartition(high_part).repartition(low_part),
            },
            {
                "name": "coal",
                "desc": "Coalesce to decrease partitions",
                "transform": lambda df: df.repartition(high_part).coalesce(low_part),
            },
        ]

        for config in test_configs:
            start_time = time.time()

            if self.pipe_type == "df":
                # For DataFrame implementation
                orig_df = self.pipe.parsed_df
                self.pipe.parsed_df = config["transform"](self.pipe.parsed_df)
                self.pipe.group(mode=False).count()
                self.pipe.parsed_df = orig_df
            else:
                # For RDD implementation
                if hasattr(self.pipe, "parsed_rdd"):
                    orig_rdd = self.pipe.parsed_rdd
                    self.pipe.parsed_rdd = config["transform"](self.pipe.parsed_rdd)
                    result = self.pipe.group(mode=False)
                    if hasattr(result, "count"):
                        result.count()
                    self.pipe.parsed_rdd = orig_rdd

            self.test_metrics["t3_" + config["name"]] = time.time() - start_time
            print(
                f"  {config['desc']}: {self.test_metrics['t3_' + config['name']]:.4f} seconds"
            )

        # Record totals
        self.test_metrics["t3_tot"] = time.time() - t3_start_time
        self.mem_metrics("t3")

        print(f"Total test time: {self.test_metrics['t3_tot']:.4f} seconds\n")
        return self.test_metrics

    def run_all_tests(self):
        """Run all performance tests in sequence."""
        print_section(f"PERFORMANCE TESTS - {self.pipe_type.upper()} IMPLEMENTATION")

        start_time = time.time()

        # Run all tests in sequence
        tests = [
            self.test_partition_count,
            self.test_repartitioning,
            self.test_coalesce_vs_repartition,
        ]

        for test_func in tests:
            test_func()

        self.test_metrics["all_tests_time"] = time.time() - start_time
        print(
            f"All tests completed in {self.test_metrics['all_tests_time']:.4f} seconds"
        )

        return self.test_metrics

In [ ]:
# Create test instances for both implementations
df_tester = PerformanceTester(df_pipe, spark)
rdd_tester = PerformanceTester(rdd_pipe, spark)

# Run all tests for DataFrame implementation
df_test_metrics = df_tester.run_all_tests()

# Run all tests for RDD implementation
rdd_test_metrics = rdd_tester.run_all_tests()

# Display combined metrics
print_section("PERFORMANCE COMPARISON: DataFrame vs RDD")
print("DataFrame Implementation Metrics:")
df_tester.display_metrics()

print("\nRDD Implementation Metrics:")
rdd_tester.display_metrics()


PERFORMANCE TESTS - DF IMPLEMENTATION

TEST 1: Partition Count Testing (DF)


+--------------------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+-----+
|             BlockId|              events|           intervals|latency|              levels|          components|                pids|Label|
+--------------------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+-----+
|blk_-100000252996...|[E5, E5, E5, E22,...|[0, 0, 0, 0, 2, 0...|      3|[INFO, INFO, INFO...|[dfs.DataNode$Dat...|[25370, 25851, 25...|    0|
|blk_-100000266894...|[E22, E5, E5, E5,...|[0, 0, 0, 0, 35, ...|  30958|[INFO, INFO, INFO...|[dfs.FSNamesystem...|[31, 5852, 6104, ...|    0|
|blk_-100000729289...|[E5, E5, E22, E5,...|[0, 0, 0, 1, 1, 0...|      2|[INFO, INFO, INFO...|[dfs.DataNode$Dat...|[25843, 26131, 28...|    0|
|blk_-100001458415...|[E5, E22, E5, E5,...|[0, 1, 1, 1, 38, ...|  31460|[INFO, INFO, INFO...|[dfs.DataNode$Dat...|[5887, 26, 5980, ...|    0|
|blk_-

  Partitions: 1, Time: 217.4240 seconds


+--------------------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+-----+
|             BlockId|              events|           intervals|latency|              levels|          components|                pids|Label|
+--------------------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+-----+
|blk_-100000729289...|[E5, E5, E22, E5,...|[0, 0, 0, 1, 1, 0...|      2|[INFO, INFO, INFO...|[dfs.DataNode$Dat...|[25843, 26131, 28...|    0|
|blk_-100002865877...|[E5, E5, E5, E22,...|[0, 0, 0, 0, 17, ...|  24078|[INFO, INFO, INFO...|[dfs.DataNode$Dat...|[15750, 15869, 15...|    0|
|blk_-100005457728...|[E22, E5, E5, E5,...|[0, 0, 1, 0, 45, ...|  33057|[INFO, INFO, INFO...|[dfs.FSNamesystem...|[30, 5268, 5177, ...|    0|
|blk_-100009528570...|[E22, E5, E5, E5,...|[0, 0, 0, 0, 34, ...|  36773|[INFO, INFO, INFO...|[dfs.FSNamesystem...|[34, 9453, 9461, ...|    0|
|blk_-

  Partitions: 2, Time: 50.5824 seconds


...

  Baseline (no repartitioning): 564.7965 seconds

Sample of grouped block data:
+------------------------+--------------------+--------------------+-------+--------------------+-----+
|BlockId                 |events              |intervals           |latency|components          |Label|
+------------------------+--------------------+--------------------+-------+--------------------+-----+


|blk_-5850359028677005163|['E5', 'E5', 'E22'] |[0, 0, 0]           |51303  |['dfs.DataNode$DataXceiver', 'dfs.DataNode$DataXceiver', 'dfs.FSNamesystem']|0    |
|blk_5587819217601489167 |['E22', 'E5', 'E5'] |[0, 0, 0]           |1637   |['dfs.FSNamesystem', 'dfs.DataNode$DataXceiver', 'dfs.DataNode$DataXceiver']|0    |
|blk_-6006351640752836388|['E5', 'E5', 'E22'] |[0, 0, 0]           |47     |['dfs.DataNode$DataXceiver', 'dfs.DataNode$DataXceiver', 'dfs.FSNamesystem']|0    |
|blk_1756405765835991300 |['E5', 'E22', 'E5'] |[0, 0, 0]           |47206  |['dfs.DataNode$DataXceiver', 'dfs.FSNamesystem', 'dfs.DataNode$DataXceiver']|0    |
|blk_-1447492252611531335|['E5', 'E5', 'E5']  |[0, 0, 0]           |48032  |['dfs.DataNode$DataXceiver', 'dfs.DataNode$DataXceiver', 'dfs.DataNode$DataXceiver']|0    |
+------------------------+--------------------+--------------------+-------+--------------------+-----+


ERROR:root:KeyboardInterrupt while sending command.               (0 + 10) / 16]
Traceback (most recent call last):
  File "/Users/farzanmirza/miniconda3/envs/pyspark_env/lib/python3.9/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/Users/farzanmirza/miniconda3/envs/pyspark_env/lib/python3.9/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/Users/farzanmirza/miniconda3/envs/pyspark_env/lib/python3.9/socket.py", line 716, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt
25/03/12 14:10:23 ERROR Executor: Exception in task 7.0 in stage 280.0 (TID 1691)
org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/opt/spark/python/lib/pyspark.zip/pyspark/worker.py", line 830, in main
    process()
  File "/opt/spark/python/lib/pyspark.zip/pyspark/worker.py", line 820, in process
    out_iter = func(s

KeyboardInterrupt: 

Traceback (most recent call last):
  File "/opt/spark/python/lib/pyspark.zip/pyspark/daemon.py", line 193, in manager
  File "/opt/spark/python/lib/pyspark.zip/pyspark/daemon.py", line 74, in worker
  File "/opt/spark/python/lib/pyspark.zip/pyspark/worker.py", line 874, in main
    if read_int(infile) == SpecialLengths.END_OF_STREAM:
  File "/opt/spark/python/lib/pyspark.zip/pyspark/serializers.py", line 596, in read_int
    raise EOFError
EOFError
25/03/12 14:10:26 WARN TaskSetManager: Lost task 15.0 in stage 280.0 (TID 1699) (macbookpro executor driver): TaskKilled (Stage cancelled)
25/03/12 14:10:27 WARN PythonRunner: Incomplete task 10.0 in stage 280 (TID 1694) interrupted: Attempting to kill Python Worker
25/03/12 14:10:27 WARN PythonRunner: Incomplete task 13.0 in stage 280 (TID 1697) interrupted: Attempting to kill Python Worker
25/03/12 14:10:27 WARN TaskSetManager: Lost task 13.0 in stage 280.0 (TID 1697) (macbookpro executor driver): TaskKilled (Stage cancelled)
25/03/12 14:1